<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Estudiar por Capítulos, Contar Votos entre Varios y Guardar Palabras en Cajones 🗂️
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Big Data
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 03
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Big%20Data/03%20-%20Analitica%20y%20Visualizacion%20en%20Big%20Data/Para%20Dummies/01_Tecnicas_Analitica_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 01 del Módulo 03 (Técnicas de analítica: ML, aprendizaje profundo y NLP)](../01_Tecnicas_de_Analitica_ML_DL_NLP.ipynb). Cuando los datos son **enormes**, los programas que *aprenden* tienen que trabajar de otra forma, igual que una persona que debe **estudiar para un examen enorme**:

| En la vida diaria | En los datos |
|---|---|
| **Estudiar un capítulo cada noche** en vez de leerse todo el libro en una sentada | Aprender **por tandas**: el modelo ve **un trozo cada vez** |
| **Cuatro ayudantes** que cuentan votos cada uno en su montón y luego se juntan los resultados | **Repartir el trabajo** entre varias máquinas y combinar resultados **con cuidado** |
| **Un archivador con 8 cajones** donde cada palabra va siempre al mismo cajón, sin libro de índices | El **truco del *hashing***: palabras → números sin guardar un vocabulario |

*(Este cuaderno **no** usa Spark ni Java: todo corre en Python con pandas.)* Al terminar podrás explicar:
1. Por qué estudiar **por tandas** casi iguala a estudiar todo junto, y por qué **el orden importa**.
2. Por qué al juntar los conteos de varios ayudantes **hay que pesar por el tamaño de cada montón**.
3. Cómo funcionan los **cajones** y qué pasa cuando **dos palabras caen en el mismo**.

> 🔗 Si luego quieres ver las versiones **con Spark, una red neuronal escrita desde cero y texto en español**: [cuaderno estándar](../01_Tecnicas_de_Analitica_ML_DL_NLP.ipynb). Antes de esto: [Don Álvaro y sus cuatro preguntas](00_Introduccion_Cap2_Dummies.ipynb).

---
## 1. Estudiar un Capítulo Cada Noche 📚

Un comprador de papa quiere aprender a **distinguir una papa buena de una mala** mirando dos cosas: el **peso** y cuántas **manchas** tiene (datos **inventados**: 3 000 papas). Comparamos dos formas de aprender:

- **Todo junto:** el modelo ve las 3 000 papas a la vez (como leerse todo el libro).
- **Por tandas:** ve **6 tandas de 500** papas, una después de la otra, y **va ajustando** lo aprendido (un capítulo por noche).

Y una **trampa**: ¿qué pasa si las tandas llegan **ordenadas** (primero todas las buenas y luego todas las malas)?

In [ ]:
import zlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)
n = 3000
peso = rng.normal(150, 30, n)                       # gramos
manchas = rng.poisson(3, n).astype(float)
prob_buena = 1 / (1 + np.exp(-(0.04 * (peso - 150) - 0.9 * (manchas - 3))))
buena = (rng.random(n) < prob_buena).astype(int)    # 1 = papa buena
papas = pd.DataFrame({"peso": peso, "manchas": manchas, "buena": buena})
entrena, prueba = papas.iloc[:2400], papas.iloc[2400:]
esc = StandardScaler().fit(entrena[["peso", "manchas"]])
X, y = esc.transform(entrena[["peso", "manchas"]]), entrena.buena.to_numpy()
Xp, yp = esc.transform(prueba[["peso", "manchas"]]), prueba.buena.to_numpy()

junto = LogisticRegression().fit(X, y)
acc_junto = junto.score(Xp, yp)

def por_tandas(orden):
    modelo = SGDClassifier(loss="log_loss", learning_rate="constant", eta0=0.01, random_state=0)
    for trozo in np.array_split(orden, 6):                          # 6 tandas de 400 papas
        modelo.partial_fit(X[trozo], y[trozo], classes=[0, 1])
    return modelo.score(Xp, yp)

acc_tandas = por_tandas(rng.permutation(len(y)))                    # tandas MEZCLADAS
acc_ordenado = por_tandas(np.argsort(y, kind="stable"))             # primero todas las malas, luego todas las buenas
print(pd.Series({"todo junto": acc_junto, "por tandas (mezcladas)": acc_tandas, "por tandas (ordenadas por clase)": acc_ordenado}).round(3).to_string())

assert abs(acc_junto - acc_tandas) < 0.03                          # por tandas casi iguala
assert acc_ordenado < acc_tandas - 0.05                            # el orden malo estropea lo aprendido

---
**Lección:** estudiar **por tandas** casi iguala a estudiar todo junto (por eso se usa cuando el libro **no cabe en la mesa**), **siempre que las tandas lleguen mezcladas**. Si llegan **ordenadas** (primero todo lo malo, luego todo lo bueno), el modelo **se acuerda solo de lo último que vio**, como el estudiante que repasa un tema hasta el final y se olvida del resto. Por eso con datos enormes **se revuelve el orden** o se leen las tandas **al azar**.

---

## 2. Cuatro Ayudantes Cuentan Votos 🗳️

En una elección de **la junta de la vereda** se cuentan las papeletas con **4 ayudantes**; cada uno cuenta **su montón** y anota **cuántas papeletas tenía** y **cuántas eran para la candidata A**. Los montones **no son iguales**: 10, 30, 60 y 100 papeletas. ¿Cuál es el **porcentaje de A** en todas las papeletas? Dos formas de juntar los resultados:

- **Sumar** votos de A y papeletas de cada ayudante y dividir.
- **Promediar** los porcentajes de los 4 ayudantes (sin pensar en el tamaño de cada montón).

In [ ]:
ayudantes = pd.DataFrame({"ayudante": ["Pedro", "Lucía", "Marta", "Jaime"], "papeletas": [10, 30, 60, 100], "votos_A": [9, 20, 24, 40]})
ayudantes["porcentaje_A"] = ayudantes.votos_A / ayudantes.papeletas
print(ayudantes.to_string(index=False))

correcto = ayudantes.votos_A.sum() / ayudantes.papeletas.sum()                              # sumar y luego dividir
promedio_simple = ayudantes.porcentaje_A.mean()                                             # promediar los porcentajes
ponderado = (ayudantes.porcentaje_A * ayudantes.papeletas).sum() / ayudantes.papeletas.sum()   # promediar PESANDO por el tamano de cada monton
print(f"\nSumando votos y papeletas: {correcto:.1%} | promedio simple de los 4 porcentajes: {promedio_simple:.1%} | promedio pesado por tamaño: {ponderado:.1%}")

assert abs(correcto - ponderado) < 1e-12                       # pesar por el tamano da el resultado correcto
assert abs(correcto - promedio_simple) > 0.05                  # el promedio simple se equivoca por mas de 5 puntos

---
**Lección:** Pedro contó solo **10** papeletas y le salió **90 %** para A, pero su montón **pesa muy poco**. Si promedias los cuatro porcentajes **como si todos pesaran lo mismo**, A parece tener **casi 59 %** cuando en realidad tiene **≈ 46.5 %**. La forma correcta es **sumar lo contado por cada uno y dividir al final** (o pesar cada porcentaje por el tamaño de su montón). Así se hace cuando varias **máquinas** reparten el trabajo de entrenar un modelo: cada una trabaja con **su trozo de datos** y los resultados se juntan **pesando por el tamaño del trozo**.

---

## 3. El Archivador de 8 Cajones 🗄️

Para clasificar textos hay que convertir **palabras en números**. Una forma es llevar un **libro de índices** con todas las palabras (pesado si hay millones). El **truco del *hashing*** es usar un archivador de **pocos cajones** y una **regla fija** que, dada una palabra, dice **siempre el mismo cajón** (sin libro). El costo: a veces **dos palabras distintas caen en el mismo cajón** (*colisión*). Probemos con **40 palabras** y archivadores de 8, 64 y 1 024 cajones.

In [ ]:
palabras = ("papa cebolla tomate zanahoria arveja frijol maíz haba yuca plátano mora fresa lulo curuba uchuva granadilla tomate_de_árbol "
            "mandarina naranja limón aguacate mango piña papaya guayaba banano durazno pera manzana ciruela uva cilantro perejil ajo apio lechuga repollo "
            "espinaca brócoli coliflor").split()
assert len(palabras) == 40


def cajon(palabra, cajones):
    return zlib.crc32(palabra.encode("utf-8")) % cajones           # regla FIJA: la misma palabra, siempre el mismo cajon


print("Con 8 cajones:", {p: cajon(p, 8) for p in palabras[:6]}, "... (cada palabra siempre cae en el mismo cajón)")
assert all(cajon(p, 8) == cajon(p, 8) for p in palabras)

filas = []
for m in (8, 64, 1024):
    cajones = pd.Series([cajon(p, m) for p in palabras])
    solas = (cajones.map(cajones.value_counts()) == 1).sum()          # palabras que NO comparten cajon
    filas.append((m, len(palabras) - solas, cajones.value_counts().max()))
tabla = pd.DataFrame(filas, columns=["cajones", "palabras que comparten cajón", "máximo de palabras en un cajón"]).set_index("cajones")
print("\n" + tabla.to_string())
assert tabla.iloc[0, 0] > tabla.iloc[1, 0] >= tabla.iloc[2, 0]        # mas cajones, menos colisiones
assert tabla.iloc[0, 0] >= 30

---
**Lección:** con **8 cajones**, casi todas las palabras **se amontonan** (hay muchísimas colisiones); con **64**, bastantes menos; con **1 024**, **casi ninguna**. Elegir el número de cajones es **un compromiso**: **pocos** cajones ahorran memoria pero **mezclan** palabras distintas (el computador las confunde); **muchos** cajones se parecen a tener el libro de índices. Lo bueno: **no hace falta guardar el vocabulario**, funciona con palabras nuevas, y cualquier máquina del grupo puede usarlo sin ponerse de acuerdo con las demás.

---

## Resumen en una frase 🎯

> **Con datos enormes, los modelos estudian por tandas (mezcladas), los resultados parciales de varias máquinas se juntan pesando por el tamaño de cada trozo, y las palabras se guardan en cajones con una regla fija, aceptando alguna colisión.**

### Lo que aprendiste hoy:
- ✅ **Por tandas:** casi igual de bueno que todo junto, **si las tandas llegan mezcladas**; ordenadas por clase, el modelo olvida.
- ✅ **Juntar resultados:** sumar votos y papeletas (o pesar por el tamaño del montón); promediar porcentajes sin pesar **se equivoca**.
- ✅ **Cajones (*hashing*):** sin libro de índices y con una regla fija; más cajones, menos colisiones.

> 🎓 **Siguiente paso:** [el cuaderno estándar](../01_Tecnicas_de_Analitica_ML_DL_NLP.ipynb) lo hace con **200 mil filas, *Spark*, una red neuronal y texto en español**. Para seguir con los gráficos: [Mil puntos en un mapa](02_Herramientas_Visualizacion_Dummies.ipynb).

---
##### 🛠️ Práctica 1: ¿Cuántos cajones necesito?

Usa la lista `palabras` (40 frutas y verduras) y la función `cajon(palabra, cajones)` de arriba:

1. Para cada número de cajones en `[8, 16, 32, 64, 128, 256]` calcula la **fracción de palabras que comparten cajón** con otra y guárdala en el diccionario `fraccion`.
2. Encuentra en `minimo` el **menor** número de cajones para el que esa fracción sea **menor o igual a 0.20**.
3. Comprueba con `assert` que `fraccion[minimo] <= 0.20` y que para el número de cajones anterior de la lista la fracción es **mayor** que 0.20.

In [ ]:
# Escribe tu código aquí

# opciones = [8, 16, 32, 64, 128, 256]
# fraccion = {}
# for m in opciones:
#     ...
# minimo = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
opciones = [8, 16, 32, 64, 128, 256]
fraccion = {}
for m in opciones:
    cajones = pd.Series([cajon(p, m) for p in palabras])
    comparten = (cajones.map(cajones.value_counts()) > 1).sum()
    fraccion[m] = comparten / len(palabras)
minimo = next(m for m in opciones if fraccion[m] <= 0.20)
anterior = opciones[opciones.index(minimo) - 1]
print({m: round(float(f), 2) for m, f in fraccion.items()}, "| mínimo:", minimo)

assert fraccion[minimo] <= 0.20
assert fraccion[anterior] > 0.20
print("Con 40 palabras hacen falta MUCHOS mas cajones que palabras para que casi nadie comparta: la paradoja del cumpleanos.")
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué es mala idea que los datos lleguen **ordenados por clase** cuando el modelo aprende **por tandas**? ¿Qué harías antes de empezar?
2. En la junta de la vereda, si Pedro hubiera contado 1 000 papeletas en vez de 10, ¿el promedio simple se habría equivocado menos? ¿Por qué?
3. ¿En qué se parece el archivador de cajones a una **bodega** con estantes numerados? ¿Qué se pierde cuando **dos cosas distintas** van al mismo estante?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Big Data (Edición Didáctica Para Dummies)</i>
  </p>
</div>